# Train your Nullified AI model (free, on Kaggle)

Runs on Kaggle's **free P100 GPU (16 GB)** — enough VRAM for gpt-oss-20B where Colab's T4 (14.5 GB) isn't.

**Before running:** in the right-hand panel set **Accelerator → GPU P100** and **Internet → On** (Internet needs a one-time phone verification on your Kaggle account). Then **Run All**.

You need a Hugging Face **Write** token (https://huggingface.co/settings/tokens); paste it when asked. Takes ~30–60 min and saves `nullified-coder` + a GGUF to your Hugging Face.

In [ ]:
#  1. Check the GPU (want: Tesla P100, 16 GB)
!nvidia-smi -L
!nvidia-smi --query-gpu=memory.total --format=csv

In [ ]:
#  2. Install + get the code (a few minutes)
!pip install -q unsloth 'huggingface_hub>=0.26'
!git clone -q https://github.com/surgeclient/nullified-ai.git /kaggle/working/nullified-ai || (cd /kaggle/working/nullified-ai && git pull -q)
print('ready')

In [ ]:
#  3. Your Hugging Face token (Write access)
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face WRITE token (hf_...): ').strip()
from huggingface_hub import HfApi
print('signed in as', HfApi(token=os.environ['HF_TOKEN']).whoami()['name'])

In [ ]:
#  4. Settings
RUNS = 'repos3'
ADAPTER_REPO = 'nullified-coder'
EPOCHS = 1.0  # 1 pass over the ~650 augmented examples finishes in ~2h and avoids overfitting
MAX_SEQ = 4096   # P100 has more room; try 6144 to keep more mods if this succeeds
RANK = 16
print('will train on runs:', RUNS)

In [ ]:
#  5. Train + upload (the long step)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!cd /kaggle/working/nullified-ai && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python training/train_lora.py --runs "$RUNS" --adapter-repo "$ADAPTER_REPO" --epochs $EPOCHS --max-seq-train $MAX_SEQ --rank $RANK --attn-only --gguf

When it finishes it prints your model repo (ending `-gguf`). Use it on the website's Build tab under **Options → Model**.